# Assignment 2 – Task 1: Titanic Passenger Survival Prediction System
---
**Deployment Link:** [Titanic Predictor on Modal](https://mhassan0000--titanic-predictor-run.modal.run)

This notebook follows the complete Machine Learning lifecycle:
1. Data Collection
2. Data Understanding (EDA)
3. Data Pre-processing & Label Encoding
4. Model Training (SVC)
5. Model Evaluation
6. Making Predictions
7. Saving the Trained Model

**Dataset:** Titanic (built-in seaborn dataset)
**Algorithm:** Support Vector Classifier (SVC)


## Step 0 – Import Libraries

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
warnings.filterwarnings("ignore")
print("Libraries imported successfully")

## Step 1 – Data Collection
Load the Titanic dataset (seaborn built-in).

In [ ]:
df = sns.load_dataset("titanic")
print(f"Dataset Shape: {df.shape}")
df.head(10)

## Step 2 – Data Understanding (EDA)

In [ ]:
print("=== Dataset Info ===")
df.info()

In [ ]:
print("=== Descriptive Statistics ===")
df.describe()

In [ ]:
print("=== Missing Values ===")
df.isnull().sum()

In [ ]:
print("=== Survival Distribution ===")
print(df["survived"].value_counts())

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Titanic – Exploratory Data Analysis", fontsize=14, fontweight="bold")

sns.countplot(data=df, x="survived", ax=axes[0,0], palette="Set2")
axes[0,0].set_title("Survival Count (0=No, 1=Yes)")

sns.countplot(data=df, x="sex", hue="survived", ax=axes[0,1], palette="Set1")
axes[0,1].set_title("Survival by Sex")

sns.countplot(data=df, x="pclass", hue="survived", ax=axes[1,0], palette="Set3")
axes[1,0].set_title("Survival by Passenger Class")

df["age"].dropna().hist(bins=30, ax=axes[1,1], color="steelblue", edgecolor="black")
axes[1,1].set_title("Age Distribution")

plt.tight_layout()
plt.savefig("titanic_eda.png", dpi=100)
plt.show()
print("EDA plots saved")

## Step 3 – Data Pre-processing & Label Encoding

In [ ]:
# Select relevant features
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target   = "survived"

df_model = df[features + [target]].copy()
df_model.dropna(inplace=True)
print(f"Shape after dropping nulls: {df_model.shape}")

# Label Encoding
le_sex      = LabelEncoder()
le_embarked = LabelEncoder()

df_model["sex"]      = le_sex.fit_transform(df_model["sex"])
df_model["embarked"] = le_embarked.fit_transform(df_model["embarked"])

# Save encoded dataset
df_model.to_csv("titanic_encoded.csv", index=False)
print("Encoded dataset saved to titanic_encoded.csv")
df_model.head()

## Step 4 – Model Training (Support Vector Classifier)

In [ ]:
X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training samples : {X_train.shape[0]}")
print(f"Testing  samples : {X_test.shape[0]}")

svc = SVC(kernel="rbf", random_state=42)
svc.fit(X_train, y_train)
print("\nSVC model trained successfully")

## Step 5 – Model Evaluation

In [ ]:
y_pred = svc.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
cm     = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc*100:.2f}%")
print("\nConfusion Matrix:")
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Did Not Survive","Survived"],
            yticklabels=["Did Not Survive","Survived"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Titanic – Confusion Matrix")
plt.tight_layout()
plt.savefig("titanic_confusion_matrix.png", dpi=100)
plt.show()

## Step 6 – Making Predictions on Sample Data

In [ ]:
sample_raw = pd.DataFrame({
    "pclass":   [1,    3,    2],
    "sex":      ["female", "male", "female"],
    "age":      [29,   22,   35],
    "sibsp":    [0,    1,    1],
    "parch":    [0,    0,    0],
    "fare":     [211,  7.25, 26],
    "embarked": ["S",  "S",  "C"],
})
sample = sample_raw.copy()
sample["sex"]      = le_sex.transform(sample["sex"])
sample["embarked"] = le_embarked.transform(sample["embarked"])

predictions = svc.predict(sample)
sample_raw["Predicted_Survival"] = ["Survived" if p==1 else "Did Not Survive"
                                     for p in predictions]

# Save predictions
sample_raw.to_csv("titanic_predictions.csv", index=False)
print("Predictions saved to titanic_predictions.csv")
sample_raw

## Step 7 – Save Trained Model

In [ ]:
with open("titanic_svc_model.pkl", "wb") as f:
    pickle.dump(svc, f)

print("Model saved to titanic_svc_model.pkl")
print("\n" + "="*50)
print("TASK 1 COMPLETE")
print("="*50)